# 14章 アプリケーションとデータ転送

本章では、以下の2つの重要なトピックについて学習します：

1. パケットの待ち行列制御とスケジューリング
2. ファイル転送プロトコル（FTP）の実装と動作

前半では、通信機器におけるパケットの転送順序について扱い、後半では実際のアプリケーション層プロトコルの実装例としてFTPを学習します。

## 待ち行列

### 待ち行列とは

スイッチやルータなどの通信機器におけるパケットの転送順序について扱います。
前章までは複数の通信装置間で連携するためのプロトコルや手順を扱ってきましたが、
ここで一度、通信装置内部の動作の理解に移行します。

一つの通信ノードや回線を多数のアプリケーションが共用するとき、
リアルタイム性の高い通信を優先させたり、公平に扱ったりしたい，というケースがあります。
このような要求に応えるため、パケットの待ち行列制御が重要になります。

In [ ]:
import sys

!git clone https://github.com/flyby-yunakayama/network-simulator.git
sys.path.insert(0,'/content/network-simulator')
%cd network-simulator

from sec14a.NetworkEventScheduler import NetworkEventScheduler
from sec14a.Node import Node
from sec14a.Switch import Switch
from sec14a.Router import Router
from sec14a.Server import DNSServer, DHCPServer
from sec14a.Link import Link
from sec14a.Application import DnsClient, DhcpClient, UDPApp, FTPClient, FTPServer

# ... (rest of the simulation code) ...

### 演習課題1: 優先制御の確認

#### 目的
パケットの優先度による完全優先制御の挙動の確認。

#### 課題の概要
トラフィックのDSCP値を増減させながら、パケットを送受信してみましょう。
DSCP値が高い、すなわち優先度が高いパケットが優先的に転送され、優先度が低いパケットの遅延が増大することを確認しましょう。

### 演習課題2: ラウンドロビンキューの実装

#### 目的
ラウンドロビンキューをシミュレータに実装し、公平制御について理解する。

#### 課題
完全優先制御の代わりに、ラウンドロビンでパケットを読み出して出力するキューを実装してみましょう。

## ファイル転送プロトコル（FTP）

本セクションでは、ファイル転送プロトコル（FTP）の基本的な仕組みと、シミュレータでの実装について学習します。

### FTPの概要

FTP（File Transfer Protocol）は、インターネット上でファイルを転送するための基本的なプロトコルです。
ネットワークアプリケーションの中でも最も基本的で重要な機能の一つとして、以下のような用途で広く使われています：

1. Webサイトのコンテンツアップロード
2. データのバックアップと共有
3. ソフトウェアの配布
4. サーバー間のファイル同期

### FTPの基本アーキテクチャ

FTPは典型的なクライアント・サーバモデルを採用しています：

1. **FTPサーバ**
   - ファイルを保管・管理
   - クライアントからの要求を待ち受け
   - アクセス制御とセキュリティ管理

2. **FTPクライアント**
   - サーバへの接続要求
   - ファイルのアップロード/ダウンロード
   - ユーザーインターフェースの提供

### シミュレータでの実装について

実際のFTPプロトコルは非常に複雑で、以下のような機能を持っています：
- 制御用とデータ転送用の2つのTCPコネクション
- 様々なファイル操作コマンド（GET, PUT, LIST等）
- バイナリ/テキストモードの転送
- 認証とアクセス制御

しかし、本シミュレータでは学習を容易にするため、以下のように簡略化しています：
- 1つのTCPコネクションでの通信
- 基本的なファイル取得（GET）のみをサポート
- シンプルなエラー処理
- 認証機能の省略

この簡略化により、ネットワークプロトコルの基本的な仕組みに焦点を当てて学習できます。

## FTPClientとFTPServerの実装

このセクションでは、シミュレータにおけるFTPクライアントとサーバの実装について説明します。実際のFTPプロトコルを学習用に簡略化し、基本的なファイル転送の仕組みを理解しやすくしています。

### FTPClientクラスの実装

FTPClientクラスは、クライアント側のFTP機能を実装します。以下に主要な部分を示します：

```python
class FTPClient:
    def __init__(self, node, server_url=None, verbose=False):
        self.node = node
        self.app_manager = node.application_layer
        self.server_url = server_url
        self.verbose = verbose
        self.state = "NOT_CONNECTED"
        self.file_to_retrieve = None
```

このコンストラクタでは：
- `node`: ネットワークノードへの参照
- `server_url`: 接続先FTPサーバのURL
- `verbose`: 詳細なログ出力の有無
- `state`: クライアントの状態管理
を初期化します。

接続処理は以下のメソッドで行われます：

```python
def connect(self, server_ip, server_port=21):
    if self.verbose:
        print("[FTPClient] Requesting TCP connect to ", server_ip, server_port)
    self.server_ip = server_ip
    self.server_port = server_port
    self.state = "CONNECTING"
    self.node.initiate_tcp_handshake(server_ip, server_port)
    self.app_manager.map_connection_to_app((server_ip, server_port), "FTP")
```

接続処理では：
1. サーバのIPアドレスとポート番号を保存
2. 状態を"CONNECTING"に設定
3. TCPハンドシェイクを開始
4. アプリケーションマネージャに接続を登録
という手順で実行されます。

### FTPServerクラスの実装

FTPServerクラスは、サーバ側のFTP機能を実装します：

```python
class FTPServer:
    def __init__(self, node, shared_files, verbose=False):
        self.node = node
        self.app_manager = node.application_layer  
        self.shared_files = shared_files
        self.verbose = verbose
        self.state = "READY"
```

サーバの初期化では：
- `shared_files`: 共有するファイルのディクショナリ
- `state`: サーバの状態管理
などを設定します。

ファイルリクエストの処理は以下のように実装されています：

```python
def on_packet_received(self, packet):
    data = packet.payload.decode('utf-8', errors='ignore')
    if data.startswith("RETR"):
        filename = parts[1]
        file_data = self.shared_files.get(filename, None)
        if file_data is None:
            self.send_ftp_response(client_ip, client_port, server_port, 
                                 "550 File not found.\r\n")
            return
        
        # ファイル転送の準備
        file_size = len(file_data)
        transfer_info = {
            'file_size': file_size,
            'bytes_transferred': 0,
            'transfer_done': False
        }
        self.node.tcp_connections[connection_key]['transfer_info'] = transfer_info
        self.outgoing_data[connection_key] = file_data
```

ファイル転送処理では：
1. クライアントからのRETRコマンドを受信
2. 要求されたファイルの存在確認
3. 転送情報の初期化
4. ファイルデータの準備
という手順で実行されます。

### 簡略化のポイント

本実装では、学習を容易にするため以下の点を簡略化しています：

1. **認証処理**: 実際のFTPでは複雑な認証が必要ですが、ここでは簡略化されています
2. **コマンド**: 基本的なファイル取得（RETR）のみをサポート
3. **データ接続**: 制御用とデータ用の別々のTCP接続を使用せず、1つの接続で処理
4. **エラー処理**: 基本的なエラーチェックのみを実装

これらの簡略化により、FTPの基本的な仕組みに焦点を当てて学習できます。

## FTPの動作確認

このセクションでは、実際にFTPクライアントとサーバを使用してファイル転送を行う方法を説明します。

### 基本的なセットアップ

まず、FTPサーバとクライアントの設定を行います：

```python
### FTPサーバの設定
shared_files = {
    "example.txt": b"Hello, this is a test file content!",
    "data.bin": b"Binary file content example"
}
ftp_server = FTPServer(node2, shared_files, verbose=True)

### FTPクライアントの設定
ftp_client = FTPClient(node1, verbose=True)
```

この例では：
- サーバ側で2つのファイル（example.txt と data.bin）を共有するように設定
- verboseモードを有効にして、詳細なログを出力するように設定

### ファイル転送の実行

FTPクライアントからサーバへの接続とファイル転送は以下のように行います：

```python
### サーバへの接続
ftp_client.connect("192.168.2.1", 21)

### ファイルの取得
ftp_client.retrieve_file("example.txt")
```

### 実行結果の例

以下は、FTPクライアントとサーバ間でファイル転送を行った際のログ出力例です：

```
[FTPClient] Requesting TCP connect to 192.168.2.1 21
[FTPClient] Connection established. Waiting for server greeting (220)...
[FTPServer] Connection established. Sending 220 greeting.
[FTPServer] Sending response: 220 Service ready
[FTPClient] Received: 220 Service ready
[FTPClient] Sending command: USER anonymous
[FTPServer] Received: USER anonymous
[FTPServer] Sending response: 331 User name okay, need password.
[FTPClient] Received: 331 User name okay, need password.
[FTPClient] Sending command: PASS anonymous@
[FTPServer] Received: PASS anonymous@
[FTPServer] Sending response: 230 User logged in, proceed.
[FTPClient] Received: 230 User logged in, proceed.
[FTPClient] Sending command: RETR example.txt
[FTPServer] Received: RETR example.txt
[FTPServer] Sending response: 150 File status okay; about to open data connection.
[FTPServer] Transfer complete. Sent 226 response.
```

### 動作の解説

1. **接続の確立**:
   - クライアントがサーバに接続要求を送信
   - サーバが応答して接続を受け入れ

2. **認証プロセス**:
   - クライアントが匿名ユーザとしてログイン
   - サーバが認証を受け入れ

3. **ファイル転送**:
   - クライアントがRETRコマンドでファイルを要求
   - サーバがファイルを送信
   - 転送完了後に確認メッセージを送信

このように、FTPクライアントとサーバ間でファイル転送が行われる様子を確認できます。verboseモードを使用することで、通信の詳細な流れを理解することができます。

## 演習課題

このセクションでは、FTPクライアントとサーバの理解を深めるための演習課題を提供します。

### 演習課題1：基本的なファイル転送

**目的**
- FTPクライアントの基本的な使い方を理解する
- サーバへの接続方法を学ぶ
- ファイル転送の仕組みを確認する

**課題内容**
1. 以下のコードを参考に、FTPサーバに新しいファイル（"test.txt"）を追加し、クライアントからそのファイルを取得してみましょう：

```python
### サーバ側の準備
shared_files = {
    "test.txt": b"This is a test file for exercise 1"
}
ftp_server = FTPServer(node2, shared_files, verbose=True)

### クライアント側の操作
ftp_client = FTPClient(node1, verbose=True)
ftp_client.connect("192.168.2.1", 21)
ftp_client.retrieve_file("test.txt")
```

**確認ポイント**
- サーバへの接続は成功しましたか？
- ファイルの転送は完了しましたか？
- ログ出力から通信の流れを確認できましたか？

### 演習課題2：複数ファイルの転送

**目的**
- 複数ファイルの転送方法を理解する
- ファイル転送の順序を考える
- エラー処理の重要性を学ぶ

**課題内容**
1. 以下のように複数のファイルを用意し、順番に転送してみましょう：

```python
### サーバ側の準備
shared_files = {
    "file1.txt": b"Content of file 1",
    "file2.txt": b"Content of file 2",
    "file3.txt": b"Content of file 3"
}
ftp_server = FTPServer(node2, shared_files, verbose=True)

### クライアント側の操作
ftp_client = FTPClient(node1, verbose=True)
ftp_client.connect("192.168.2.1", 21)

### 複数ファイルの転送
files_to_retrieve = ["file1.txt", "file2.txt", "file3.txt"]
for filename in files_to_retrieve:
    ftp_client.retrieve_file(filename)
```

**確認ポイント**
- すべてのファイルが正しく転送されましたか？
- 転送の順序は意図した通りですか？
- エラーが発生した場合、適切に処理されましたか？

### 演習課題3：エラー処理の確認

**目的**
- エラー発生時の動作を理解する
- 存在しないファイルへのアクセス時の挙動を確認する
- デバッグの重要性を学ぶ

**課題内容**
1. 存在しないファイルにアクセスしてエラーの挙動を確認してみましょう：

```python
### サーバ側の準備
shared_files = {
    "existing.txt": b"This file exists"
}
ftp_server = FTPServer(node2, shared_files, verbose=True)

### クライアント側の操作
ftp_client = FTPClient(node1, verbose=True)
ftp_client.connect("192.168.2.1", 21)

### 存在しないファイルへのアクセス
ftp_client.retrieve_file("nonexistent.txt")
```

**確認ポイント**
- エラーメッセージは適切に表示されましたか？
- クライアントはエラー後も正常に動作しますか？
- ログから何が起きたかを理解できましたか？

### 発展課題：ネットワーク構成の変更

**目的**
- ネットワークトポロジの影響を理解する
- IPアドレス設定の重要性を学ぶ
- 実践的なトラブルシューティングを経験する

**課題内容**
1. 以下のように異なるネットワーク構成で接続を試してみましょう：

```python
### 新しいネットワーク構成の作成
node3 = Node(node_id="n3", ip_address="192.168.3.1/24", network_event_scheduler=nes)
router2 = Router(node_id="r2", 
                ip_addresses=["192.168.2.253/24", "192.168.3.254/24"],
                network_event_scheduler=nes)

### 新しいリンクの追加
link6 = Link(node3, router2, bandwidth=100000, delay=0.01, loss_rate=0.0,
            network_event_scheduler=nes)
link7 = Link(router2, router1, bandwidth=100000, delay=0.01, loss_rate=0.0,
            network_event_scheduler=nes)

### FTPクライアントの設定
ftp_client = FTPClient(node3, verbose=True)
ftp_client.connect("192.168.2.1", 21)
```

**確認ポイント**
- 異なるサブネット間でも接続できましたか？
- ルーティングは正しく機能していますか？
- 遅延やパケットロスの影響は確認できましたか？

これらの演習を通じて、FTPの基本的な動作から応用的な使用方法まで理解を深めることができます。また、ネットワークの基本的な概念も同時に学ぶことができます。

## 本章のまとめ

本章では、パケットのキューイングとスケジューリング、そしてファイル転送プロトコル（FTP）について扱いました。


前半では、待ち行列とキューおよびスケジューラの概念を学習し、シンプルなシミュレーションによって待ち時間やキュー長について学びました。
また優先制御やラウンドロビンといった基本的な手法についても確認しました。
その上で、優先度を表すフィールドをパケットに付与できるようにシミュレータをアップデートし、優先制御を実装しました。

後半では、アプリケーション層プロトコルの一つであるFTPの基本的な仕組みを学習しました。
実際のFTPの複雑な機能を学習用に簡略化したFTPクライアントとサーバを実装し、
基本的なファイル転送の仕組みについて理解を深めました。

これらの学習を通じて、ネットワークアプリケーションの基本的な動作と、
その上で動作する各種プロトコルの役割について理解できたかと思います。


本章では、アプリケーション層のプロトコルとして、ファイル転送プロトコル（FTP）について学習しました。

### 学習内容の要点

1. **FTPの基本概念**
   - クライアント・サーバモデルによるファイル転送の仕組み
   - 制御接続とデータ接続の役割（本シミュレータでは簡略化）
   - ファイル転送の基本的な手順

2. **シミュレータでの実装**
   - FTPClientクラスとFTPServerクラスによる基本機能の実現
   - シンプルな実装による学習しやすさの重視
   - 実際のFTPプロトコルとの違いと簡略化のポイント

3. **他の章との関連**
   - 第12章で学んだDHCPによるIPアドレスの自動設定
   - DNSによるホスト名解決の仕組み（www.example.comなどのURLをIPアドレスに変換）
   - TCPによる信頼性の高い通信の実現

### 実装の特徴

本シミュレータのFTP実装では、学習効果を高めるため、以下の点を重視しました：

1. **シンプルな設計**
   - 複雑な認証処理を省略
   - 基本的なファイル転送機能に焦点
   - クラス変数による直接的なデータ管理

2. **可視性の向上**
   - verboseモードによる通信過程の可視化
   - ログ出力による動作確認の容易さ
   - 段階的な処理の理解のしやすさ

3. **実践的な学習環境**
   - 実際のネットワーク環境に近い構成
   - エラー処理の基本的な実装
   - 様々なネットワーク構成での動作確認

### 発展的な学習に向けて

本章で学んだ内容を基に、以下のような発展的な学習が可能です：

1. **プロトコルの拡張**
   - より複雑なファイル操作の実装
   - セキュリティ機能の追加
   - 並行処理の実現

2. **ネットワーク構成の応用**
   - 複数のルータを経由した転送
   - 帯域制限がある環境での動作
   - 様々なエラー状況への対応

このように、FTPの基本的な仕組みを理解することで、より高度なネットワークアプリケーションの開発や運用に必要な知識を身につけることができます。

In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here


In [ ]:
# Example code cell
# Add relevant code examples here
